# Task 2 — Measure and compare

## Save and submit / שמירה והגשה
1. **File → Save a copy in Drive** before editing. Rename with your student code and task.
2. Write answers in text cells. Save outputs. Runtime files are temporary: download data/images separately.
3. Save, close and reopen your copy. Verify that your answer and plot remain.
4. Share with the mentor account given in class, normally as **Commenter**. Send the link through the agreed private channel.
5. Keep a dated `.ipynb` download if you need a fixed submission snapshot.

שמרו עותק אישי, פתחו אותו מחדש ובדקו שהתשובות נשמרו. שיתוף מחברת אינו שיתוף אוטומטי של קובצי סביבת ההרצה.


**At home: allow 60–90 minutes; split into two sessions if useful.** Save your own copy as `YOUR-CODE_task-2`. Follow the [numbered Task 2 instructions](https://yaminbens.github.io/alpha-mentorship/task-2.html) in another tab. Run four fresh preparations: 0, 8, 24 and 8 Faster clicks. Keep N=64, box side=12, gravity=0 and time step=0.005. Each run has 1000 preparation steps, 1000 settling steps and 40 readings separated by 25 steps. This notebook does the arithmetic; you collect the data.

Double-click text cells to replace `…`. In the results code, replace only `None` with your measured numbers. You may answer in English or Hebrew.

# Task 2 report · Measure and compare
Fill this in inside your Task 2 notebook, or copy it into your own document.
Replace `…` with your words. Missing measurements should stay blank; do not invent values.
English or Hebrew is welcome. A short, clear answer is enough.

Student code: …   Date: …   Notebook/task version: …
Data source: my own runs / provided examples (name the source): …

## A. Question and prediction — before running
Question: how does adding energy affect motion, arrangement and potential energy
per particle, when particle number and box size stay fixed?
My prediction: I think … because … .
I will check it by measuring … and observing … .

## B. Method — record what you actually did
- Simulator/model and boundaries: …
- Initial-state filename: …
- Particle count N: …; box side: …; time step: …
- Gravity, bonds, anchored particles, fixed-temperature particles: …
- Preparation: … steps. Settling after Faster: … steps.
- Measurement: … readings, … steps between readings, from time … to time … .
- Units: reduced model units; temperature is not in kelvin.
- What I kept fixed: …
- Any different setting, missed click, warning or other problem: …



## C. Calculate one run’s averages
Paste the complete tab-separated export, including headings, into the empty triple quotes in the next cell. Run it once per file. Copy the two averages into the matching row of **My results** before replacing the pasted text with the next file. Keep all four raw files separately; this cell holds only the most recently pasted file.

KE is kinetic energy, PE is potential energy, N is particle count. For this 2D model the helper averages KE/N and PE/N over your readings. It does not use the displayed running-average T column. Units are the model’s own reduced units, not kelvin or joules.

In [ ]:
import io
import numpy as np
import pandas as pd

# Paste ONE complete System totals / More detail table between these triple quotes.
# Keep the column headings and tabs. Save each original file separately.
browser_tsv = """
"""

def average_browser_run(text):
    raw = pd.read_csv(io.StringIO(text.strip()), sep="\t")
    raw.columns = raw.columns.str.strip()
    required = ["t", "N", "KE", "PE"]
    missing = [key for key in required if key not in raw.columns]
    if missing:
        raise ValueError("Missing columns: " + ", ".join(missing) +
                         ". Choose System totals and More detail; copy the header too.")
    values = raw[required].apply(pd.to_numeric, errors="raise")
    if values.empty or not np.isfinite(values.to_numpy()).all():
        raise ValueError("The table has empty or non-numeric measurements. Check the raw file.")
    if (values.N <= 0).any():
        raise ValueError("Particle count N must be positive. Check the export.")
    flags = []
    if len(values) != 40:
        flags.append("Expected 40 readings; found " + str(len(values)))
    if not (values.N == 64).all():
        flags.append("Particle count differs from the planned N=64")
    if len(values) != 40 or not np.allclose(values.t, np.arange(1,41)*0.125, atol=0.001):
        flags.append("Times differ from the planned 0.125, 0.250, ..., 5.000 window")
    return {"readings": len(values), "T_reduced": float((values.KE/values.N).mean()),
            "U_per_particle": float((values.PE/values.N).mean()), "flags": flags}

if not browser_tsv.strip():
    print("Paste one run's raw table above, then run this cell again. No averages yet.")
else:
    try:
        result = average_browser_run(browser_tsv)
        print("Readings:", result["readings"])
        print("T_reduced =", result["T_reduced"])
        print("U_per_particle =", result["U_per_particle"])
        for flag in result["flags"]:
            print("FLAGGED:", flag, "— keep the data and explain the difference in your method.")
        print("Copy these two averages into the matching results row below.")
    except (ValueError, pd.errors.ParserError) as error:
        print("Could not calculate averages:", error)
        print("Keep your original file and ask your mentor if you need help.")


## My results
Each row below is one run. Replace its two `None` entries with the two averages printed above. Leave unmeasured entries as `None`; the empty template should not produce results. After filling all four rows, run this cell. Example of the syntax only: replace `None` with a number using a decimal point. Do not copy demonstration values as your own observations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Replace only None with YOUR measured averages. Use a decimal point, not a comma.
# Each line describes one run. Keep the labels and punctuation.
rows = [
    {"condition": "low",           "repeat": 1, "T_reduced": None, "U_per_particle": None},
    {"condition": "middle",        "repeat": 1, "T_reduced": None, "U_per_particle": None},
    {"condition": "high",          "repeat": 1, "T_reduced": None, "U_per_particle": None},
    {"condition": "middle-repeat", "repeat": 2, "T_reduced": None, "U_per_particle": None},
]
table = pd.DataFrame(rows)
print(table.to_string(index=False))


## D. Make my graph
Run the next cell after **My results**. The repeat can lie exactly on the middle point. This is possible when replaying the same original state. If a warning appears, keep it with your notes.

In [ ]:
if table[["T_reduced", "U_per_particle"]].isna().any().any():
    print("Enter your four pairs of averages above, then rerun My results and this cell.")
    print("No result plotted yet. Missing data should remain blank.")
else:
    for column in ["T_reduced", "U_per_particle"]:
        table[column] = pd.to_numeric(table[column], errors="raise")
    if not np.isfinite(table[["T_reduced", "U_per_particle"]].to_numpy()).all():
        print("A value is not finite. Check the raw data and record the problem; no graph saved.")
    else:
        fig, ax = plt.subplots()
        for index, row in table.iterrows():
            ax.scatter(row.T_reduced, row.U_per_particle, marker="x" if index==3 else "o")
            ax.annotate(row["condition"], (row.T_reduced, row.U_per_particle),
                        xytext=(5, -14 if index==3 else 6), textcoords="offset points")
        ax.set_xlabel("Measured k_B T / epsilon (reduced temperature)")
        ax.set_ylabel("U / (N epsilon) (potential energy per particle)")
        ax.set_title("My four recorded runs; the repeat may overlap the middle point")
        fig.tight_layout()
        fig.savefig("my_energy_plot.png", dpi=160, bbox_inches="tight")
        plt.show()
        table.to_csv("my_observations.csv", index=False)
        print("Created my_energy_plot.png and my_observations.csv. Download both.")


## Save the graph and results file
In Colab, click the folder icon in the left sidebar. Refresh the file list if needed. Open the three-dot menu beside `my_energy_plot.png`, choose **Download**, and do the same for `my_observations.csv`. Store both in your shared work folder. These files exist only after a complete table has been plotted. Runtime files disappear when the session is removed.

My four raw data files and screenshots are saved here: …

Motion and arrangement in low: …

Motion and arrangement in middle: …

Motion and arrangement in high: …

The middle repeat agrees/differs in these ways: …

As measured temperature increases, potential energy per particle … .

This graph does not directly measure arrangement. An additional measurement could be … .

## E. Conclusion — about 150 words
I changed … and kept … fixed.
I observed … . The numbers or observations supporting this are … .
This agrees/disagrees with my prediction because … .
Replaying the same starting state tells me …, but it does not tell me … .
I am still unsure about … . Next I would measure … .

## F. Save, submit and revise
- [ ] Four result rows, graph and conclusion are in my saved notebook/document.
- [ ] Four raw files, screenshots, graph and results CSV are saved separately.
- [ ] I reopened the notebook and checked the contents.
- [ ] My mentor has access to the notebook and supporting files.
- [ ] I sent the link through the agreed private channel.

If unfinished, the step where I need help is … .
After feedback: on [date] I changed … because … .


## Interpretation reminder
A same-state replay checks the procedure; it is not an estimate of uncertainty across independent preparations. A short observation and an energy plot do not establish an equilibrium melting temperature. Keep unexpected results. If using provided data, name its source explicitly.

## Reflection and revision
What did you learn? What remains uncertain? What changed after feedback?

**Your answer:** …